# Gradient Descent trên dữ liệu giá nhà

*Bài 5, mục 5.3.1, Bước 3.* Vai trò của chuẩn hóa đặc trưng, so sánh Batch, SGD, Mini-batch với nghiệm OLS.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA = Path('..') / 'data'
df = pd.read_csv(DATA / 'house_prices.csv')
X = df[['area', 'bedrooms', 'distance']].values   # diện tích, số phòng ngủ, khoảng cách
y = df['price'].values                             # giá nhà (triệu VND)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


def add_ones(X):
    return np.column_stack([np.ones(len(X)), X])


def loss(w, Xb, y):
    return np.sum((Xb @ w - y)**2) / (2 * len(y))


def grad(w, Xb, y):
    return Xb.T @ (Xb @ w - y) / len(y)


def minibatch_gd(Xb, y, eta, batch_size, n_epochs, w_init=None, seed=0):
    rng = np.random.default_rng(seed)
    N = len(y)
    w = np.zeros(Xb.shape[1]) if w_init is None else w_init.copy()
    history = [loss(w, Xb, y)]                    # loss trên toàn bộ dữ liệu sau mỗi epoch
    for epoch in range(n_epochs):
        order = rng.permutation(N)               # xáo trộn dữ liệu mỗi epoch
        for s in range(0, N, batch_size):
            B = order[s:s + batch_size]          # chỉ số các mẫu trong mini-batch
            w = w - eta * grad(w, Xb[B], y[B])    # gradient trung bình trên mini-batch
        history.append(loss(w, Xb, y))
    return w, history


def evaluate(w, Xb, y):
    pred = Xb @ w
    rmse = np.sqrt(np.mean((pred - y)**2))
    r2 = 1 - np.sum((pred - y)**2) / np.sum((y - y.mean())**2)
    return rmse, r2

## 1. Nghiệm OLS để đối chiếu

In [2]:
Xb_train = add_ones(X_train)
w_ols = np.linalg.pinv(Xb_train.T @ Xb_train) @ Xb_train.T @ y_train
print('OLS: w = %s, RMSE test = %.2f, R2 = %.4f'
      % (w_ols.round(2), *evaluate(w_ols, add_ones(X_test), y_test)))

OLS: w = [505.2   29.98 144.89 -59.46], RMSE test = 302.06, R2 = 0.9685


## 2. GD trên dữ liệu gốc (chưa chuẩn hóa)

Các đặc trưng có thang đo chênh lệch lớn: $\eta = 3 \times 10^{-4}$ đã phân kỳ, $\eta = 10^{-4}$ thì hội tụ cực kỳ chậm.

In [3]:
with np.errstate(over='ignore', invalid='ignore'):
    w, history = minibatch_gd(Xb_train, y_train, eta=3e-4, batch_size=len(y_train), n_epochs=50)
print('eta = 3e-4: loss =', np.round(history[:4], 1), '... -> phân kỳ')
w, history = minibatch_gd(Xb_train, y_train, eta=1e-4, batch_size=len(y_train), n_epochs=1000)
print('eta = 1e-4, 1000 vòng lặp: w =', w.round(2), '-> còn rất xa OLS')

eta = 3e-4: loss = [7.90193940e+06 1.03334795e+08 1.37383378e+09 1.82874400e+10] ... -> phân kỳ


eta = 1e-4, 1000 vòng lặp: w = [  4.46  35.79   8.73 -45.2 ] -> còn rất xa OLS


## 3. Chuẩn hóa đặc trưng rồi so sánh ba biến thể

Chỉ dùng trung bình và độ lệch chuẩn của tập huấn luyện. Sau khi huấn luyện, đổi trọng số về thang đo gốc: $w_j = w'_j / \sigma_j$, $w_0 = w'_0 - \sum_j w'_j \mu_j / \sigma_j$.

In [4]:
mu, std = X_train.mean(axis=0), X_train.std(axis=0)
Zb_train = add_ones((X_train - mu) / std)
Zb_test = add_ones((X_test - mu) / std)

for name, batch_size, eta, n_epochs in [('Batch', len(y_train), 0.5, 100),
                               ('SGD', 1, 0.01, 10),
                               ('Mini-batch', 64, 0.1, 10)]:
    w, history = minibatch_gd(Zb_train, y_train, eta, batch_size, n_epochs)
    w_original = np.r_[w[0] - np.sum(w[1:] * mu / std), w[1:] / std]
    rmse, r2 = evaluate(w, Zb_test, y_test)
    print('%-10s loss = %.1f, w (thang gốc) = %s, RMSE test = %.2f, R2 = %.4f'
          % (name, history[-1], w_original.round(2), rmse, r2))

Batch      loss = 45584.0, w (thang gốc) = [505.26  29.98 145.1  -59.46], RMSE test = 302.05, R2 = 0.9685


SGD        loss = 46949.9, w (thang gốc) = [647.97  29.73 140.1  -65.22], RMSE test = 305.79, R2 = 0.9677
Mini-batch loss = 45657.7, w (thang gốc) = [519.97  29.98 147.92 -60.75], RMSE test = 302.16, R2 = 0.9685
